# 3 · Spectral clustering of haemoglobin

Companion to **Section 5**. The contact network of human deoxyhaemoglobin
(PDB 2HHB): 574 residues, joined when the separation between their
C$_\alpha$ atoms is 4–8 Å.
The molecule is a tetramer (chains A, C = $\alpha$; B, D = $\beta$), so the
external four-chain reference partition is available independently of the
graph construction.

In [ ]:
import sys
sys.path.insert(0, "../code")          # the scripts that generate the paper's figures
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
np.set_printoptions(precision=4, suppress=True)


def draw(ax, P, edges, nodes_kw=None, title=None):
    """Draw a graph with node positions P (n x 2) and an edge list."""
    ax.add_collection(LineCollection([[P[i], P[j]] for i, j in edges],
                                     colors="0.75", linewidths=0.6, zorder=1))
    ax.scatter(P[:, 0], P[:, 1], s=14, zorder=2, **(nodes_kw or {}))
    ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)

In [ ]:
from collections import Counter
from scipy.spatial.distance import pdist, squareform
from common import graph_clustering, spectra, SEED
from fig4_clustering import kmeans, ncut, best_match, pca2
G, A, d, L, chain, xyz = graph_clustering()
lam, Q, Lnorm, lam_n, Q_n = spectra(L, d)
n = len(d)
print(n, "residues,", G.number_of_edges(), "contacts, chains:", dict(Counter(map(str, chain))))

## How many clusters? The eigengap rule

$k^\star=\arg\max_{k\ge2}\{\lambda_{k+1}(L_{\rm norm})-\lambda_k(L_{\rm norm})\}$, searched over the prespecified coarse range $2\le k\le 9$.

In [ ]:
gaps = np.diff(lam_n[:11])
k_star = int(np.argmax(gaps[1:9]) + 2)
plt.plot(range(1, 11), lam_n[:10], "o-"); plt.xlabel("i"); plt.ylabel("lambda_i(L_norm)")
plt.title(f"k* = {k_star}"); plt.show()
print("gaps after k = 2..9:", gaps[1:9])

## Embed, normalize the rows, run k-means

The symmetric normalized spectral-clustering algorithm uses the first $k$ eigenvectors
$U=[\mathbf u_1,\dots,\mathbf u_k]$ of $L_{\rm norm}$, including the trivial first
eigenvector. Normalize each row of $U$ to unit length before running k-means.

In [ ]:
U = Q_n[:, :k_star]
U = U / np.linalg.norm(U, axis=1, keepdims=True)
lab, _ = kmeans(U, k_star, np.random.default_rng(SEED))
acc, mapping = best_match(lab, chain, k_star)
print(f"agreement with the chains: {acc:.3f}   Ncut = {ncut(lab, A, d):.3f}")
# name each cluster after the chain it matches best, so that colours agree
inv = {q: c for c, q in mapping.items()}
lab_chain = np.array([inv.get(q, "E") for q in lab])   # "E": cluster matched to no chain (k* > 4)
for c in "ABCD":
    print("chain", c, "->", dict(Counter(map(str, lab_chain[chain == c]))))

P = pca2(xyz)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, colour, name in [(ax[0], [ord(c) - 65 for c in chain], "true chains"),
                        (ax[1], [ord(c) - 65 for c in lab_chain], "spectral clusters")]:
    a.scatter(*P.T, c=colour, cmap="tab10", vmin=0, vmax=9, s=5)
    a.set_aspect("equal"); a.axis("off"); a.set_title(name)
plt.show()

## Is $k^\star=4$ robust?

`contact_network(cutoff)` rebuilds the graph from the residue coordinates with a
different upper cut-off. Exercise 7: loop over 6, 6.5, …, 10 Å and record
connectivity, $k^\star$ and the agreement obtained with $k=4$.

In [ ]:
Dist = squareform(pdist(xyz))

def contact_network(cutoff, lower=4.0):
    Ac = ((Dist >= lower) & (Dist <= cutoff)).astype(float)
    np.fill_diagonal(Ac, 0.0)
    return Ac

Ac = contact_network(7.0)
dc = Ac.sum(1)
print("7 A:", int(Ac.sum() / 2), "contacts, isolated residues:", int((dc == 0).sum()))